# Temporal validation

Shows why the project splits by time rather than randomly: the split boundaries, per-split fraud rates, and the absence of time overlap between train, validation, and test windows.

**Prerequisites:** `make sample-data`, `make ingest`, and `make features`.

In [1]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [2]:
from pyspark.sql import functions as F

from transaction_risk.models.split import temporal_split
from transaction_risk.spark.io import read_table

features = read_table(spark, '../data/gold/features')
train_df, validation_df, test_df = temporal_split(features)

for name, split_df in [('train', train_df), ('validation', validation_df), ('test', test_df)]:
    summary = split_df.agg(
        F.min('step').alias('min_step'),
        F.max('step').alias('max_step'),
        F.count(F.lit(1)).alias('rows'),
        F.avg(F.col('isFraud').cast('double')).alias('fraud_rate'),
    ).collect()[0]
    print(
        f"{name:>10}: steps {summary['min_step']:>4} – {summary['max_step']:>4}, "
        f"rows {summary['rows']:>7}, fraud rate {summary['fraud_rate']:.4f}"
    )

     train: steps    0 –  174, rows    3500, fraud rate 0.0143


validation: steps  175 –  211, rows     740, fraud rate 0.0270


      test: steps  212 –  249, rows     760, fraud rate 0.0158


The three windows are disjoint in time: the model trains on the earliest steps, selects its threshold on the next window, and is evaluated on the most recent one. A random split would leak future account behaviour into training through the historical aggregate features, inflating offline metrics relative to what production would see.

In [3]:
spark.stop()